# Bản mini: ghi -> train -> chấm, chạy ở máy nhà trong ~10 phút

Đúng 3 bước như notebook Colab (`colab/so101_act_colab.ipynb`), nhưng mọi thứ thu nhỏ để **hiểu cách chạy**, không để ra mạng giỏi:

| Bước | Ở đây | Bản thật trên Colab |
|---|---|---|
| 1. Ghi dataset | 3 ván | 500 ván + 800 ván DAgger |
| 2. Train | 300 step, batch 4 | 60 000 step, batch 32 (~3,7 giờ trên L4) |
| 3. Chấm | 2-3 seed | 774 seed |

Mạng mini chỉ train 300 step nên **sẽ gắp hỏng**: đó là bình thường. Ô 3b chấm thử một checkpoint thật có sẵn ở `D:/vla` để so sánh.
Ô 4 mở "hộp đen": tự viết vòng lặp nhìn -> đoán -> làm, ~15 dòng, để thấy mạng nhận gì và trả gì.

**Chạy:** mở file này trong VS Code, chọn kernel `D:/vla/venv-lerobot/Scripts/python.exe`, chạy lần lượt từng ô (Shift+Enter).
Mọi file sinh ra nằm ở `D:/vla/mini/` (xoá thoải mái).

In [ ]:
# ===== Ô 0: chuẩn bị =====
# Mục đích: có sẵn vài biến đường dẫn và một hàm run() để gọi các script của dự án, giống ô "!python -m ..." trên Colab.

# --- Thư viện chuẩn của Python (không cần cài) ---
import os          # os.environ: đọc / sửa biến môi trường (dùng để ép output tiếng Việt in đúng)
import shutil      # shutil.rmtree: xoá cả một thư mục (xoá dataset / checkpoint cũ trước khi tạo lại)
import subprocess  # subprocess.Popen: chạy một chương trình khác (ở đây: python -m sim_engine...) và đọc output của nó
import sys         # sys.executable: đường dẫn tới python.exe đang chạy notebook này
from pathlib import Path   # Path: đường dẫn dạng đối tượng; ghép bằng dấu "/" thay vì cộng chuỗi, chạy đúng cả Windows lẫn Linux

# --- Tìm thư mục dự án ---
# sim_engine được cài kiểu "editable" (pip install -e): import ở đâu cũng được, và nó trỏ thẳng vào thư mục sim-engine/ trên máy,
# sửa code xong không cần cài lại.
import sim_engine
# sim_engine.__file__ = '.../VLA/sim-engine/__init__.py'; .parent = thư mục chứa file đó = gốc repo.
# Cần REPO để tìm file config YAML ở ô 2 mà không phụ thuộc notebook đang đứng ở thư mục nào.
REPO = Path(sim_engine.__file__).parent

# MINI: mọi thứ bản mini sinh ra (dataset, checkpoint) dồn vào một chỗ trên ổ D (ổ C gần đầy), xoá cả thư mục là sạch.
MINI = Path('D:/vla/mini')

# PY: dùng ĐÚNG python của kernel (D:/vla/venv-lerobot) để chạy các script con. Nếu gõ "python" trơn, Windows có thể chọn
# một python khác không có LeRobot -> lỗi "No module named lerobot".
PY = sys.executable


def run(module, *args, keep=lambda line: True):
    """Chạy `python -m <module> <args...>` và in output từng dòng ngay khi có (không đợi chạy xong mới in).

    module: tên module kiểu dấu chấm, vd 'sim_engine.learning.record_dataset' (= file sim-engine/learning/record_dataset.py)
    *args:  các tham số dòng lệnh, vd '--episodes', 3. Số / Path được đổi sang chuỗi bên dưới.
    keep:   hàm nhận một dòng output, trả True nếu muốn in dòng đó. Mặc định in hết. Ô train dùng để chỉ giữ dòng loss.
    """
    # Copy toàn bộ biến môi trường hiện tại ({**os.environ}) rồi thêm PYTHONIOENCODING=utf-8: script con in chữ tiếng Việt ("HỎNG")
    # mà Windows mặc định dùng bảng mã cp1258 sẽ báo UnicodeEncodeError. Biến này ép script con xuất UTF-8.
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    # Popen khởi động tiến trình con và trả về NGAY (không chờ). Danh sách lệnh: [python.exe, '-m', module, tham số...].
    #   map(str, args): đổi từng tham số sang chuỗi (subprocess chỉ nhận chuỗi; 3 -> '3', Path -> 'D:\\vla\\...').
    #   stdout=PIPE: lấy output của script con về đây để lọc và in, thay vì in thẳng ra (notebook sẽ không thấy).
    #   stderr=STDOUT: gộp luồng lỗi vào luồng output thường, để thấy cả thông báo lỗi theo đúng thứ tự.
    #   text=True + encoding='utf-8': đọc output thành chuỗi UTF-8 (không phải bytes).
    #   errors='replace': gặp byte lạ thì thay bằng dấu "?" chứ không dừng cả lệnh.
    p = subprocess.Popen([PY, '-m', module, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace', env=env)
    # Đọc lần lượt từng dòng script con in ra; vòng lặp kết thúc khi script con chạy xong và đóng output.
    for line in p.stdout:
        # noise: những dòng không giúp hiểu gì: cảnh báo của thư viện ('warn'), log của bộ mã hoá video SVT-AV1 ('Svt[', '[mp4'),
        # thanh tiến độ của thư viện datasets ('Map:'). Lọc đi cho output gọn.
        noise = 'warn' in line.lower() or line.startswith(('Svt[', '[mp4', 'Map:'))
        if not noise and keep(line):
            print(line, end='')   # line đã có sẵn ký tự xuống dòng ở cuối, end='' để không xuống dòng hai lần
    # p.wait(): chờ tiến trình con kết thúc hẳn, trả về mã thoát. 0 = thành công; khác 0 = có lỗi.
    # Ném lỗi để notebook dừng ở ô này (đỡ chạy tiếp ô sau với dữ liệu thiếu).
    if p.wait() != 0:
        raise RuntimeError(f'{module} lỗi (mã {p.returncode})')


# --- Kiểm tra môi trường ---
import torch   # PyTorch: thư viện mạng nơ-ron mà LeRobot dùng; ở đây chỉ để hỏi có GPU không
print('repo:', REPO)                                  # phải là .../VLA/sim-engine
# torch.cuda.is_available(): có GPU NVIDIA + driver + PyTorch bản CUDA không. Không có thì train / chấm chạy CPU, chậm gấp chục lần.
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có (sẽ rất chậm)')

## 1. Ghi dataset: thầy giáo (policy viết tay min-jerk) làm mẫu 3 ván (~1 phút)

`learning/record_dataset.py`: mỗi bước lấy ảnh 2 camera (`top`, `wrist`) + 6 góc khớp, hỏi thầy action, ghi lại thành một **khung**.
Chỉ giữ ván thành công. Seed bắt đầu từ 3000 (không trùng bộ chấm 0-2499).

In [ ]:
# ===== Ô 1: ghi dataset =====
# DS: thư mục dataset sẽ tạo. Path / 'a' / 'b' = ghép đường dẫn -> D:/vla/mini/datasets/so101_mini
DS = MINI / 'datasets' / 'so101_mini'
# Xoá dataset của lần chạy trước (nếu có): LeRobot từ chối ghi vào thư mục đã tồn tại.
# ignore_errors=True: chưa có thư mục thì thôi, không báo lỗi.
shutil.rmtree(DS, ignore_errors=True)
# Gọi learning/record_dataset.py. Bên trong nó lặp: chọn seed -> dựng cảnh -> thầy chạy cả ván, mỗi bước ghi (ảnh, góc khớp, lệnh
# thầy) -> ván thành công thì lưu, hỏng thì bỏ -> sang seed kế tiếp, cho tới khi đủ số ván.
run('sim_engine.learning.record_dataset',
    '--episodes', 3,               # cần 3 ván THÀNH CÔNG (ván hỏng không tính, thử seed tiếp)
    '--cameras', 'top,wrist',      # 2 camera giống mạng thật: trên cao (thấy cả bàn) + cổ tay (thấy kẹp lúc gần vật)
    '--root', DS,                  # ghi vào đâu
    '--repo-id', 'local/so101_mini')   # tên dataset kiểu Hugging Face "chủ/tên"; "local/..." = chỉ ở máy, không đẩy lên mạng
# Không truyền --seed-start: mặc định 3000 (sim_setup.TRAIN_SEED_START), tách khỏi seed 0-2499 dùng để chấm.

### 1b. Mở dataset xem bên trong có gì

Một khung = các trường `observation.*` (đầu vào của mạng) + `action` (đáp án của thầy). Ảnh được lưu thành video mp4, đọc ra thành
tensor `[3, cao, rộng]` giá trị 0-1.

In [ ]:
# ===== Ô 1b: mở dataset vừa ghi =====
# LeRobotDataset: lớp đọc dataset của LeRobot. Đây cũng chính là thứ lerobot-train dùng để lấy dữ liệu lúc train.
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import matplotlib.pyplot as plt   # vẽ ảnh

# Mở dataset: repo_id phải khớp lúc ghi; root = thư mục trên đĩa.
# video_backend='pyav': ảnh được lưu thành video mp4 (nhẹ hơn hàng nghìn file ảnh), cần bộ giải mã để đọc lại. Bộ mặc định
# (torchcodec) chưa chạy được trên Windows nên dùng pyav. Config train cũng đặt dataset.video_backend: pyav vì lý do này.
ds = LeRobotDataset('local/so101_mini', root=DS, video_backend='pyav')
# num_episodes: số ván; num_frames: tổng số khung của mọi ván; fps: số khung mỗi giây (= số bước điều khiển mỗi giây của sim).
print(f'{ds.num_episodes} ván, {ds.num_frames} khung, {ds.fps} khung/giây')

# features: "lược đồ" của một khung: tên trường -> kiểu dữ liệu và kích thước.
#   observation.state         6 số: đầu vào của mạng (góc khớp hiện tại)
#   observation.images.top    ảnh 240 x 320 x 3 màu: đầu vào của mạng
#   observation.images.wrist  ảnh camera cổ tay: đầu vào của mạng
#   action                    6 số: ĐÁP ÁN mạng phải học đoán (lệnh thầy ở khung này)
#   timestamp, frame_index, episode_index, index, task_index: sổ sách (khung thứ mấy, của ván nào), mạng không dùng
for name, ft in ds.features.items():
    # {name:28s}: canh trái, rộng 28 ký tự cho thẳng cột
    print(f'  {name:28s} {ft["dtype"]:8s} {ft["shape"]}')

# ds[i] = khung thứ i tính trên toàn dataset (0..num_frames-1). Ván đầu dài ~150 khung, nên khung 60 là lúc tay đang đi tới hộp.
# Kết quả là một dict: tên trường -> tensor PyTorch.
f = ds[60]
# Hai thứ khác đơn vị: state = 5 góc khớp tính bằng ĐỘ + độ mở kẹp (0 đóng .. 100 mở); action = 5 góc khớp đích tính bằng RAD +
# lệnh kẹp (> 0 mở, < 0 khép). Mạng không bận tâm: bộ chuẩn hoá đưa mọi số về cùng thang trước khi vào mạng.
# .numpy(): tensor -> mảng numpy để in; .round(1): làm tròn cho dễ đọc.
print('state  (độ, kẹp 0-100):  ', f['observation.state'].numpy().round(1))
print('action (rad, kẹp -1..1): ', f['action'].numpy().round(3))

# Vẽ 2 ảnh cạnh nhau: plt.subplots(1 hàng, 2 cột); figsize = kích thước hình (inch).
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for a, cam in zip(ax, ['top', 'wrist']):          # zip ghép từng ô vẽ với tên camera tương ứng
    img = f[f'observation.images.{cam}']          # tensor [3 màu, cao, rộng], giá trị 0-1 (kiểu PyTorch: kênh màu đứng đầu)
    a.imshow(img.permute(1, 2, 0))                # matplotlib cần [cao, rộng, 3 màu] -> permute đổi thứ tự trục
    a.set_title(cam)
    a.axis('off')                                 # bỏ trục toạ độ cho gọn

## 2. Train: ACT học bắt chước thầy, 300 step (~2 phút trên GTX 1650)

`learning/train.py` đọc file YAML (một thí nghiệm = một file trong `learning/configs/`) rồi gọi `lerobot-train`. Tham số gõ thêm
ghi đè YAML: ở đây đổi dataset, chỗ lưu, và thu nhỏ số step / batch. Đọc log theo bảng "Thuật ngữ khi train" trong `docs/so_tay_ai.md`:
`loss` phải giảm dần (chạy thử: 10,9 ở step 50 -> 2,8 ở step 300; bản thật xuống ~0,1).

In [ ]:
# ===== Ô 2: train =====
# OUT: nơi lerobot-train lưu checkpoint và log
OUT = MINI / 'outputs' / 'act_mini'
# lerobot-train báo lỗi nếu output_dir đã tồn tại (để không vô tình ghi đè một lần train dài); chỉ khi --resume=true mới cho phép.
shutil.rmtree(OUT, ignore_errors=True)
run('sim_engine.learning.train',
    # File YAML gốc: lấy MỌI tham số của thí nghiệm act_box4_topwrist (loại mạng ACT, chunk_size 50, n_action_steps 20, lr 3e-5,
    # dùng AMP...). Mở file đó ra xem: mỗi dòng có chú thích. Các tham số dưới đây GHI ĐÈ giá trị trong YAML.
    REPO / 'learning/configs/act_box4_topwrist.yaml',
    '--dataset.repo_id=local/so101_mini',   # đổi dataset sang bản mini vừa ghi (YAML trỏ tới dataset 200 ván)
    f'--dataset.root={DS}',
    f'--output_dir={OUT}',
    '--job_name=act_mini',                  # tên lần train, hiện trong log
    '--steps=300',                          # 300 lần cập nhật trọng số (bản thật 60 000)
    '--batch_size=4',                       # mỗi step học từ 4 khung ngẫu nhiên (bản thật 32; GTX 1650 4 GB vẫn chạy được 8)
    '--save_freq=150',                      # lưu checkpoint ở step 150 và 300
    '--log_freq=50',                        # in một dòng loss mỗi 50 step
    '--num_workers=0',                      # đọc dữ liệu ngay trong tiến trình chính: dataset bé, khỏi tốn thời gian mở tiến trình phụ
    # Chỉ in: dòng loss ('step:'), lỗi, và dòng lệnh lerobot-train đầy đủ mà train.py in ra đầu tiên (copy chạy tay được).
    keep=lambda line: 'step:' in line or 'Error' in line or 'lerobot-train' in line)
# Mỗi dòng log: step:50 ... loss:10.85 grdn:293 lr:3.0e-05 ... l1_loss:0.648 kld_loss:1.02
#   loss = l1_loss + 10 x kld_loss. l1_loss = sai lệch action đoán so với thầy; kld_loss = phạt phần biến ẩn (VAE) của ACT.
#   Bảng giải thích đủ các cột: docs/so_tay_ai.md, mục 7 "Thuật ngữ khi train và chấm".

# Xem train để lại gì: một thư mục cho mỗi lần lưu (000150, 000300) + 'last' (trỏ tới lần lưu mới nhất).
# iterdir(): liệt kê thư mục con; p.name: chỉ lấy tên; sorted: sắp xếp cho dễ đọc.
print(sorted(p.name for p in (OUT / 'checkpoints').iterdir()))
# Bên trong pretrained_model/ là thứ dùng để CHẠY mạng:
#   config.json                         kiến trúc mạng (loại act, kích thước, camera nào vào)
#   model.safetensors                   các trọng số đã học (~200 MB với mạng thật)
#   policy_preprocessor*.json/...       cách chuẩn hoá đầu vào (trung bình, độ lệch chuẩn tính từ dataset)
#   policy_postprocessor*.json/...      cách đổi đầu ra của mạng về thang action thật
#   train_config.json                   toàn bộ tham số lần train (để resume hoặc tra lại)
print(sorted(p.name for p in (OUT / 'checkpoints/last/pretrained_model').iterdir()))

## 3. Chấm: mạng tự lái trên vài seed của bộ chấm (không có thầy)

`learning/eval_policy.py`: mỗi bước mạng nhìn ảnh + góc khớp, ra action, mô phỏng làm theo, lặp lại (vòng kín). Mạng mini gần như
chắc chắn hỏng (thường "Chưa kẹp được").

In [ ]:
# ===== Ô 3: chấm mạng mini =====
run('sim_engine.learning.eval_policy',
    '--policy', OUT / 'checkpoints/last/pretrained_model',   # thư mục pretrained_model của checkpoint muốn chấm
    '--seeds', '0-1')                                        # chỉ seed 0 và 1 (bản thật: bỏ --seeds để chạy đủ 774 seed)
# Mỗi dòng: [ván thứ/tổng] seed X: OK / HỎNG (đã kẹp) / HỎNG, số bước (500 = hết giờ), số ván đạt cộng dồn, thời gian.
# Bảng cuối:
#   all / r<37cm / r>=37cm  tỉ lệ đạt chung / vùng gần / vùng xa (vật cách trục khớp 1 dưới / từ 37 cm)
#   grasped_but_failed      seed kẹp được vật nhưng vẫn hỏng (rơi lúc mang, thả trượt bát)
#   never_grasped           seed chưa kẹp được lần nào (tìm / tiếp cận vật kém)

### 3b. So với một checkpoint thật đã train ở nhà

`D:/vla/outputs/train/act_box4_topwrist` (200 ván, 20k step, ~36-40 %). Ô thứ hai gộp hai checkpoint của nó (trung bình action),
cùng cách đã đưa 3 mạng DAgger từ 77,8 % lên 84,8 %: chậm hơn vì mỗi bước chạy hai mạng.

In [ ]:
# ===== Ô 3b: chấm một checkpoint thật =====
# REAL: thư mục checkpoints của lần train act_box4_topwrist làm ở nhà (200 ván, 20 000 step); có 005000 ... 020000 và last.
REAL = Path('D:/vla/outputs/train/act_box4_topwrist/checkpoints')
run('sim_engine.learning.eval_policy', '--policy', REAL / 'last/pretrained_model', '--seeds', '0-2')

In [ ]:
# ===== Ô 3c: gộp hai checkpoint =====
# Truyền NHIỀU thư mục sau --policy: eval_policy nạp cả hai mạng; mỗi bước cả hai cùng đoán, action chạy = trung bình hai action
# (hàm act() trong learning/eval_policy.py). Mỗi bước chạy 2 mạng nên chậm hơn.
run('sim_engine.learning.eval_policy',
    '--policy', REAL / '010000/pretrained_model', REAL / '020000/pretrained_model',
    '--seeds', '0-2')
# Chỉ 3 seed nên hơn / kém ở đây chưa nói lên gì (sai số rất lớn); kết luận phải chấm đủ 774 seed.

## 4. Mở hộp đen: tự viết vòng lặp nhìn -> đoán -> làm

Đây chính là thân hàm `run_episode` trong `learning/eval_policy.py`, rút gọn. Chú ý cột "hàng đợi": ACT đoán 50 action một lần
(`chunk_size`), làm 20 cái (`n_action_steps`) rồi mới đoán lại; giữa hai lần đoán mạng **không nhìn ảnh**.

In [ ]:
# ===== Ô 4: tự viết vòng lặp mạng lái =====
import numpy as np                                            # mảng số (env trả / nhận numpy)
from lerobot.common.control_utils import predict_action       # hàm tiện ích của LeRobot: chuẩn hoá -> mạng -> đổi về thang thật
from sim_engine.learning import sim_setup                     # cầu nối env của ta <-> định dạng LeRobot
from sim_engine.learning.eval_policy import load_policy       # hàm nạp checkpoint (đọc nó trong eval_policy.py)

# --- Nạp mạng ---
# load_policy trả 3 thứ:
#   policy: mạng ACT (ResNet18 đọc ảnh + transformer đoán chuỗi action), đã nạp trọng số, đặt lên GPU, ở chế độ chạy (eval)
#   pre:    bộ tiền xử lý: dict numpy -> tensor GPU, chuẩn hoá (trừ trung bình, chia độ lệch chuẩn của dataset lúc train)
#   post:   bộ hậu xử lý: action mạng đoán (đã chuẩn hoá) -> đổi ngược về radian / lệnh kẹp thật
policy, pre, post = load_policy(REAL / 'last/pretrained_model', 'cuda')

# Mạng được train với camera nào thì phải đưa đúng camera đó. input_features = tên các đầu vào lưu trong config.json của checkpoint,
# vd {'observation.state', 'observation.images.top', 'observation.images.wrist'} -> cameras = ('top', 'wrist').
# env_state: True nếu mạng được train với toạ độ thật của vật (thí nghiệm "ăn gian"); mạng này thì False.
cameras, env_state = sim_setup.spec_from_features(policy.config.input_features)

# Câu lệnh chữ của nhiệm vụ. ACT không dùng nó (chỉ mạng VLA như SmolVLA mới đọc chữ), nhưng định dạng LeRobot luôn có trường này.
task = sim_setup.TASKS['box'].format(color='red')             # 'Pick up the red box and place it in the bowl'

# --- Dựng cảnh ---
# make_env(seed, loại vật, cỡ vật (m), images): đặt hộp 4 cm đúng chỗ hộp của seed 0 (cùng cảnh với replay của policy viết tay).
# options = {'bowl_xy': ...}: vị trí bát của seed, phải truyền vào reset() bên dưới.
env, options = sim_setup.make_env(0, 'box', 0.04, images=True)
# reset: đưa tay về home, đặt vật và bát. Trả:
#   obs:  quan sát đầu tiên, obs['state'] = 7 số (5 góc khớp độ, 1 ô luôn 0, độ mở kẹp)
#   info: thông tin phụ (vị trí đầu kẹp 'ee_pos', vị trí vật 'box_pos', đã thành công chưa 'success'...). Mạng KHÔNG thấy info;
#         ta chỉ dùng để in ra theo dõi.
obs, info = env.reset(seed=0, options=options)
# Xoá hàng đợi action của mạng (còn sót từ ván trước thì ván mới sẽ làm nốt action cũ).
policy.reset()

# --- Vòng kín: nhìn -> đoán -> làm, 80 bước (~2,7 giây mô phỏng; một ván đầy đủ tối đa 500 bước) ---
for t in range(80):
    # 1) NHÌN. to_frame chụp 2 camera ngay lúc này và gói cùng góc khớp thành một khung giống hệt lúc ghi dataset.
    #    np.zeros(7): chỗ cho action, lúc chạy chưa có action nên điền số 0 (bị bỏ đi ở dòng sau).
    frame = sim_setup.to_frame(env, obs, np.zeros(7), task, cameras, env_state)
    # Chỉ giữ các trường "observation.*" (đầu vào của mạng); bỏ "action" (đáp án, lúc chạy không có) và "task".
    observation = {k: v for k, v in frame.items() if k.startswith('observation.')}
    if t == 0:
        # In kích thước đầu vào một lần: state (6,), mỗi ảnh (240, 320, 3)
        print({k: getattr(v, 'shape', v) for k, v in observation.items()})

    # Hàng đợi của ACT: mỗi lần đoán, mạng ra 50 action (chunk_size) và cất 20 cái đầu (n_action_steps) vào hàng đợi.
    # Mỗi bước lấy ra 1 cái. Khi hàng đợi rỗng (queued == 0) thì bước này mạng mới thực sự nhìn ảnh và đoán chuỗi mới.
    # (_action_queue có gạch dưới = biến nội bộ của LeRobot; chỉ đọc để học, đừng dựa vào nó trong code thật.)
    queued = len(policy._action_queue)

    # 2) ĐOÁN. predict_action làm 4 việc: numpy -> tensor GPU (thêm chiều batch = 1), pre (chuẩn hoá), policy.select_action
    #    (lấy action kế tiếp trong hàng đợi, rỗng thì chạy mạng), post (đổi về thang thật). Kết quả: tensor (1, 6).
    #    use_amp=False: tính đủ 32 bit (lúc chạy không cần tăng tốc bằng 16 bit).
    action = predict_action(observation, policy, torch.device('cuda'), pre, post, use_amp=False, task=task)

    # 3) LÀM. Đưa action cho mô phỏng.
    #    action.cpu().numpy(): tensor GPU -> numpy. to_env_action: 6 số -> 7 số (env cần lệnh kẹp lặp 2 lần), cắt về [-1, 1].
    #    env.step trả 5 thứ: obs mới, reward, terminated, truncated (hết 500 bước), info. Dấu _ = bỏ, không dùng.
    #    obs mới quay lại bước 1 ở vòng sau: đó là "vòng kín", lệnh của mạng quyết định cảnh mạng thấy tiếp theo.
    obs, _, _, _, info = env.step(sim_setup.to_env_action(action.cpu().numpy()))

    # In mỗi 10 bước, và mọi bước mạng vừa đoán chuỗi mới (queued == 0).
    if t % 10 == 0 or queued == 0:
        # .ravel(): (1, 6) -> (6,) cho dễ đọc. Khoảng cách đầu kẹp - vật: np.linalg.norm = độ dài vectơ (m) -> x 100 = cm.
        print(f'bước {t:3d} | hàng đợi trước khi gọi: {queued:2d} | action {action.cpu().numpy().ravel().round(2)} '
              f'| đầu kẹp cách hộp {np.linalg.norm(info["ee_pos"] - info["box_pos"]) * 100:.1f} cm')

# Giải phóng mô phỏng và renderer (bộ nhớ đồ hoạ). Không đóng thì chạy ô này nhiều lần sẽ đầy bộ nhớ.
env.close()
# Đọc kết quả: action[5] (số cuối) > 0 là mở kẹp, < 0 là khép; khoảng cách giảm dần rồi kẹp khép khi còn ~2-3 cm.

## Đọc code theo thứ tự nào

| Thứ tự | File | Đọc để hiểu |
|---|---|---|
| 1 | `learning/sim_setup.py` | Cầu nối env <-> LeRobot: camera nào, khung gồm gì, đổi action 6 <-> 7 số, bộ seed chấm |
| 2 | `learning/record_dataset.py` | Thầy làm mẫu, chỉ giữ ván thành công (ô 1) |
| 3 | `learning/train.py` + `learning/configs/*.yaml` | Train chỉ là chọn tham số; vòng lặp thật nằm trong LeRobot (ô 2) |
| 4 | `learning/eval_policy.py` | `run_episode` (ô 4), `act` (gộp checkpoint), bảng tóm tắt gần / xa |
| 5 | `learning/dagger.py` | AI lái, thầy tiếp quản, ghi đoạn thầy sửa |
| 6 | `learning/merge_datasets.py` | Gộp nhiều dataset trước khi train |
| 7 | `colab/so101_act_colab.ipynb` | Cùng các bước, chạy nền trên GPU lớn, lưu lên Drive |

Thuật ngữ trong log và bảng kết quả: `docs/so_tay_ai.md`, mục 7.